In [1]:
import cupy as cp
import numpy as np
import pandas as pd

#Making sure a GPU is assigned
print(f"Current device name: {cp.cuda.runtime.getDeviceProperties(0)["name"]}")
mempool = cp.get_default_memory_pool()

Current device name: b'NVIDIA GeForce RTX 3070'


In [2]:
policies = pd.read_csv(r"C:\Users\pauln\OneDrive\Code\Parallelization\POLICIES.csv")
scenarios = pd.read_csv(r"C:\Users\pauln\OneDrive\Code\Parallelization\RETURN.csv")

OUTER_YEARS = 100
OUTER_SCENS = 500
NB_POLICIES = 500
NB_CALCS = 16
this_dtype = 'float64'

_T = 0
_S = 1
_POLICY = 2
_MV = 3
_RETURN = 4
_TRANSAC = 5
_GUARANTEE = 6
_CLAIMS = 0
_CALC1 = 1
_CALC2 = 2
_CALC3 = 3
_CALC4 = 4
_CALC5 = 5
_CALC6 = 6
_CALC7 = 7
_CALC8 = 8
_CALC9 = 9
_CALC10 = 10
_CALC11 = 11
_CALC12 = 12
_CALC13 = 13
_CALC14 = 14
_CALC15 = 15

OL_df = pd.merge(policies[policies.POLICY <= NB_POLICIES],scenarios,how='cross')
OL_df = OL_df[['T','S','POLICY','MV','RETURN','TRANSAC','GUARANTEE']]
#OL_df[['CLAIMS','CALC1','CALC2','CALC3','CALC4','CALC5','CALC6','CALC7','CALC8','CALC9','CALC10','CALC11','CALC12','CALC13','CALC14','CALC15']] = 0
OL_df = OL_df.sort_values(['T','S','POLICY'])

Load Numpy

In [3]:
OL_np = OL_df.to_numpy(dtype=this_dtype).reshape(OUTER_YEARS,OUTER_SCENS,NB_POLICIES,OL_df.shape[1])
OC_np = np.zeros(OL_np.shape[:-1] + (NB_CALCS,),dtype=this_dtype)

print(f"OuterLoop shape      : {OL_np.shape}")
print(f"OuterLoop memory (GB): {OL_np.nbytes/1e9}")

OuterLoop shape      : (100, 500, 500, 7)
OuterLoop memory (GB): 1.4


Loop Numpy

In [4]:
for T in range(1,OUTER_YEARS-1):
    OL_np[T][...,_MV] = OL_np[T-1][...,_MV] * ( 1 + OL_np[T][...,_RETURN]) + OL_np[T][...,_TRANSAC]
    OL_np[T][...,_GUARANTEE] = OL_np[T-1][...,_GUARANTEE]

    OC_np[T][...,_CALC1] = (OL_np[T][...,_MV] + OL_np[T][...,_GUARANTEE])/2
    OC_np[T][...,_CALC2] = (OL_np[T][...,_MV]/1000)**2
    OC_np[T][...,_CALC3] = OL_np[T][...,_MV]*( 1 + OL_np[T][...,_RETURN])**1.5
    OC_np[T][...,_CALC4] = np.nan_to_num(OL_np[T][...,_TRANSAC]/OL_np[T][...,_MV])
    OC_np[T][...,_CALC5] = 1 - ( 1 + OL_np[T][...,_RETURN])**0.5
    OC_np[T][...,_CALC6] = (OL_np[T][...,_MV] - OL_np[T][...,_GUARANTEE])/2
    OC_np[T][...,_CALC7] = (-OL_np[T][...,_MV]/1000)**2
    OC_np[T][...,_CALC8] = OL_np[T][...,_MV]*( 1 - OL_np[T][...,_RETURN])**1.5
    OC_np[T][...,_CALC9] = np.nan_to_num(OL_np[T][...,_TRANSAC]/OL_np[T][...,_MV])
    OC_np[T][...,_CALC10] = 1 - ( 1 - OL_np[T][...,_RETURN])**0.5
    OC_np[T][...,_CALC11] = (OL_np[T][...,_MV] - OL_np[T][...,_GUARANTEE])/2
    OC_np[T][...,_CALC12] = (-OL_np[T][...,_MV]/1000)**2
    OC_np[T][...,_CALC13] = OL_np[T][...,_MV]*( 1 - OL_np[T][...,_RETURN])**1.5
    OC_np[T][...,_CALC14] = np.nan_to_num(OL_np[T][...,_TRANSAC]/OL_np[T][...,_MV])
    OC_np[T][...,_CALC15] = 1 - ( 1 - OL_np[T][...,_RETURN])**0.5

    if T % 10 == 0:
        OL_np[T][...,_GUARANTEE] = np.maximum(OL_np[T-1][...,_GUARANTEE], OL_np[T][...,_MV])

    OC_np[T][...,_CLAIMS] = np.maximum((OL_np[T][...,_GUARANTEE] - OL_np[T][...,_MV]),0)


Summarize Numpy

In [5]:
NP_result = np.nansum(np.nanmean(np.nansum(OC_np[...,_CLAIMS],axis=0),axis=0),axis=0)

print(NP_result)

1446877381.6409707


Load Cupy

In [6]:
OL_cp = cp.asarray(OL_np)
OC_cp = cp.zeros(OL_cp.shape[:-1] + (NB_CALCS,),dtype=this_dtype)
print(f"Memory used by pool (GB)     : {mempool.used_bytes()/1e9}")

Memory used by pool (GB)     : 4.6


Loop Cupy

In [7]:
for T in range(1,OUTER_YEARS-1):
    OL_cp[T][...,_MV] = OL_cp[T-1][...,_MV] * ( 1 + OL_cp[T][...,_RETURN]) + OL_cp[T][...,_TRANSAC]
    OL_cp[T][...,_GUARANTEE] = OL_cp[T-1][...,_GUARANTEE]

    OC_cp[T][...,_CALC1] = (OL_cp[T][...,_MV] + OL_cp[T][...,_GUARANTEE])/2
    OC_cp[T][...,_CALC2] = (OL_cp[T][...,_MV]/1000)**2
    OC_cp[T][...,_CALC3] = OL_cp[T][...,_MV]*( 1 + OL_cp[T][...,_RETURN])**1.5
    OC_cp[T][...,_CALC4] = cp.nan_to_num(OL_cp[T][...,_TRANSAC]/OL_cp[T][...,_MV])
    OC_cp[T][...,_CALC5] = 1 - ( 1 + OL_cp[T][...,_RETURN])**0.5
    OC_cp[T][...,_CALC6] = (OL_cp[T][...,_MV] - OL_cp[T][...,_GUARANTEE])/2
    OC_cp[T][...,_CALC7] = (-OL_cp[T][...,_MV]/1000)**2
    OC_cp[T][...,_CALC8] = OL_cp[T][...,_MV]*( 1 - OL_cp[T][...,_RETURN])**1.5
    OC_cp[T][...,_CALC9] = cp.nan_to_num(OL_cp[T][...,_TRANSAC]/OL_cp[T][...,_MV])
    OC_cp[T][...,_CALC10] = 1 - ( 1 - OL_cp[T][...,_RETURN])**0.5
    OC_cp[T][...,_CALC11] = (OL_cp[T][...,_MV] - OL_cp[T][...,_GUARANTEE])/2
    OC_cp[T][...,_CALC12] = (-OL_cp[T][...,_MV]/1000)**2
    OC_cp[T][...,_CALC13] = OL_cp[T][...,_MV]*( 1 - OL_cp[T][...,_RETURN])**1.5
    OC_cp[T][...,_CALC14] = cp.nan_to_num(OL_cp[T][...,_TRANSAC]/OL_cp[T][...,_MV])
    OC_cp[T][...,_CALC15] = 1 - ( 1 - OL_cp[T][...,_RETURN])**0.5

    if T % 10 == 0:
        OL_cp[T][...,_GUARANTEE] = cp.maximum(OL_cp[T-1][...,_GUARANTEE], OL_cp[T][...,_MV])

    OC_cp[T][...,_CLAIMS] = cp.maximum((OL_cp[T][...,_GUARANTEE] - OL_cp[T][...,_MV]),0)


Summarize Cupy

In [8]:
CP_result = cp.nansum(cp.nanmean(cp.nansum(OC_cp[...,_CLAIMS],axis=0),axis=0),axis=0)

print(CP_result)

1446877381.640971
